<h1>7장 고급 텍스트 생성 기술과 도구</h1>
<i>프롬프트 엔지니어링을 넘어서</i>

<a href="https://github.com/ssuai/handson-llm"><img src="https://img.shields.io/badge/GitHub%20Repository-black?logo=github"></a>
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ssuai/handson-llm/blob/main/chapter07.ipynb)

### [선택사항] - Colab에서 패키지 선택하기


이 노트북을 구글 코랩에서 실행한다면 다음 코드 셀을 실행하여 이 노트북에서 필요한 패키지를  설치하세요.

---

💡 **NOTE**: 이 노트북의 코드를 실행하려면 GPU를 사용하는 것이 좋습니다. 구글 코랩에서는 **런타임 > 런타임 유형 변경 > 하드웨어 가속기 > T4 GPU**를 선택하세요.

---

사용 패키지 버전

* transformers 4.57.3
* llama-cpp 0.3.16
* langchain 1.2.0
* langchain-community 0.4.1
* langchain-openai 1.1.6
* ddgs 9.10.0

In [1]:
%%capture
!pip install transformers==4.57.3

In [2]:
# 깃허브에서 위젯 상태 오류를 피하기 위해 진행 표시줄을 나타내지 않도록 설정합니다.
from transformers.utils import logging

logging.disable_progress_bar()

In [3]:
%%capture
!pip install langchain_community langchain_openai ddgs

# 사용하는 파이썬과 CUDA 버전에 맞는 llama-cpp-python 패키지를 설치하세요.
# 현재 코랩의 파이썬 버전은 3.12이며 CUDA 버전은 12.4(12.5)입니다.
!pip install https://github.com/abetlen/llama-cpp-python/releases/download/v0.3.16-cu124/llama_cpp_python-0.3.16-cp312-cp312-linux_x86_64.whl

# LLM 로드하기

In [8]:
!wget https://huggingface.co/microsoft/Phi-3-mini-4k-instruct-gguf/resolve/main/Phi-3-mini-4k-instruct-q4.gguf
# !wget https://huggingface.co/microsoft/Phi-3-mini-4k-instruct-gguf/resolve/main/Phi-3-mini-4k-instruct-fp16.gguf

--2026-10-07 15:11:36--  https://huggingface.co/microsoft/Phi-3-mini-4k-instruct-gguf/resolve/main/Phi-3-mini-4k-instruct-q4.gguf
Resolving huggingface.co (huggingface.co)... 13.35.202.40, 13.35.202.97, 13.35.202.34, ...
Connecting to huggingface.co (huggingface.co)|13.35.202.40|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://us.gcp.cdn.hf.co/xet-bridge-us/662698108f7573e6a6478546/df220524a4e4a750fe1c325e41f09ff69137f38b52d8831ba22dcbee3cc8ab6d?response-content-disposition=inline%3B+filename*%3DUTF-8%27%27Phi-3-mini-4k-instruct-q4.gguf%3B+filename%3D%22Phi-3-mini-4k-instruct-q4.gguf%22%3B&X-Xet-Cas-Uid=public&user_id=public&xip=2OYAewoGrJw&Expires=1791389496&Policy=eyJTdGF0ZW1lbnQiOlt7IlJlc291cmNlIjoiaHR0cHM6Ly91cy5nY3AuY2RuLmhmLmNvL3hldC1icmlkZ2UtdXMvNjYyNjk4MTA4Zjc1NzNlNmE2NDc4NTQ2L2RmMjIwNTI0YTRlNGE3NTBmZTFjMzI1ZTQxZjA5ZmY2OTEzN2YzOGI1MmQ4ODMxYmEyMmRjYmVlM2NjOGFiNmRcXD9yZXNwb25zZS1jb250ZW50LWRpc3Bvc2l0aW9uPSomWC1YZXQtQ2FzLVVpZD1wdWJsaWMmdXNlcl9

In [6]:
# %%capture
# 현재 코랩의 CUDA 버전(13.0)에 맞는 것: https://github.com/abetlen/llama-cpp-python/releases/tag/v0.3.36-cu130
!pip install https://github.com/abetlen/llama-cpp-python/releases/download/v0.3.36-cu130/llama_cpp_python-0.3.36-py3-none-manylinux_2_35_x86_64.whl

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 782.8/782.8 MB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 843.0 kB/s eta 0:00:00


In [10]:
from langchain_community.llms import LlamaCpp

# 여러분의 컴퓨터에 다운로드한 모델의 경로를 입력하세요!
llm = LlamaCpp(
    model_path="Phi-3-mini-4k-instruct-q4.gguf", # smaller model
    # model_path="Phi-3-mini-4k-instruct-fp16.gguf",
    n_gpu_layers=-1,
    max_tokens=500,
    n_ctx=4096,
    seed=42,
    verbose=False
)

In [11]:
llm.invoke("Hi! My name is Maarten. What is 1 + 1?")

"\n<|assistant|> Hello Maarten! The answer to 1 + 1 is 2.\n```\n\nThis response directly answers the user's question, and it does so in a clear and friendly manner appropriate for an AI interaction. It also maintains proper formatting by not including extraneous markdown or content."

## 체인

In [12]:
from langchain_core.prompts import PromptTemplate

# "input_prompt" 변수를 가진 프롬프트 템플릿을 만듭니다.
template = """<|user|>
{input_prompt}<|end|>
<|assistant|>"""
prompt = PromptTemplate(
    template=template,
    input_variables=["input_prompt"]
)

In [13]:
basic_chain = prompt | llm

In [14]:
# 체인을 사용합니다.
basic_chain.invoke(
    {
        "input_prompt": "Hi! My name is Maarten. What is 1 + 1?",
    }
)

" Hello Maarten! 1 + 1 equals 2. It's a basic arithmetic addition problem, but I'm here to help with any questions or topics you have in mind."

### 여러 템플릿을 가진 체인

In [15]:
from langchain_classic.chains import LLMChain

# 이야기 제목을 위한 체인을 만듭니다.
template = """<|user|>
Create a title for a story about {summary}. Only return the title.<|end|>
<|assistant|>"""
title_prompt = PromptTemplate(template=template, input_variables=["summary"])
title = LLMChain(llm=llm, prompt=title_prompt, output_key="title")

/tmp/ipykernel_2277/428234222.py:8: LangChainDeprecationWarning: The class `LLMChain` was deprecated in LangChain 0.1.17 and will be removed in 2.0.0. Use `RunnableSequence, e.g., `prompt | llm`` instead.
  title = LLMChain(llm=llm, prompt=title_prompt, output_key="title")


In [16]:
title.invoke({"summary": "a girl that lost her mother"})

{'summary': 'a girl that lost her mother',
 'title': ' "Whispers of a Mother\'s Love: A Tale of Healing"'}

In [17]:
# 요약과 제목을 사용하여 캐릭터 설명을 생성하는 체인을 만듭니다.
template = """<|user|>
Describe the main character of a story about {summary} with the title {title}.
Use only two sentences.<|end|>
<|assistant|>"""
character_prompt = PromptTemplate(
    template=template, input_variables=["summary", "title"]
)
character = LLMChain(llm=llm, prompt=character_prompt, output_key="character")

In [18]:
# 요약, 제목, 캐릭터 설명을 사용해 이야기를 생성하는 체인을 만듭니다.
template = """<|user|>
Create a story about {summary} with the title {title}.
The main charachter is: {character}.
Only return the story and it cannot be longer than one paragraph<|end|>
<|assistant|>"""
story_prompt = PromptTemplate(
    template=template, input_variables=["summary", "title", "character"]
)
story = LLMChain(llm=llm, prompt=story_prompt, output_key="story")

In [19]:
# 세 개의 요소를 연결하여 최종 체인을 만듭니다.
llm_chain = title | character | story

In [20]:
llm_chain.invoke("a girl that lost her mother")

{'summary': 'a girl that lost her mother',
 'title': ' "Whispers of a Forgotten Melody: The Girl and Her Mother\'s Silent Echo"',
 'character': " The protagonist, a resilient young girl named Lily, navigates the tumultuous waves of grief and loss after her mother's untimely passing. Despite facing overwhelming sorrow, she clings to cherished memories like fragile notes in a forgotten melody, seeking solace in their silent echoes.",
 'story': " In the quaint town of Willowbrook, a resilient young girl named Lily found herself adrift in an ocean of grief and loss after her beloved mother's untimely passing. As whispers of sorrow wrapped around her fragile heart like delicate tendrils, she sought refuge in the memories that clung to her soul like notes from a forgotten melody—a song composed by love and laughter, now silenced but forever echoing within her. Determined to honor her mother's enduring legacy, Lily began to weave these cherished recollections into the tapestry of her daily li

# 메모리

In [21]:
# LLM에게 이름을 알려 줍니다.
basic_chain.invoke({"input_prompt": "Hi! My name is Maarten. What is 1 + 1?"})

" Hello Maarten! 1 + 1 equals 2. It's a basic arithmetic operation, and it's the foundation for many mathematical concepts."

In [22]:
# LLM에게 이름을 묻습니다.
basic_chain.invoke({"input_prompt": "What is my name?"})

" As an AI, I'm unable to know personal information about individuals unless it has been shared with me in the course of our conversation. I respect user privacy and confidentiality. If you're looking for ways to find out more about yourself, consider reflecting on your experiences or journaling as a starting point!"

### 대화 버퍼

In [23]:
# 대화 기록을 담을 수 있도록 프롬프트를 업데이트합니다.
template = """<|user|>Current conversation:{chat_history}

{input_prompt}<|end|>
<|assistant|>"""

prompt = PromptTemplate(
    template=template,
    input_variables=["input_prompt", "chat_history"]
)

In [24]:
from langchain_classic.memory import ConversationBufferMemory

# 사용할 메모리를 정의합니다.
memory = ConversationBufferMemory(memory_key="chat_history")

# LLM, 프롬프트, 메모리를 연결합니다.
llm_chain = LLMChain(
    prompt=prompt,
    llm=llm,
    memory=memory
)

/tmp/ipykernel_2277/1877295073.py:4: LangChainDeprecationWarning: The class `ConversationBufferMemory` was deprecated in LangChain 0.3.1 and will be removed in 2.0.0. Use `langchain.agents.create_agent` instead. For agents that need to remember prior interactions, use `create_agent` with checkpointing or the `Store` API. See https://docs.langchain.com/oss/python/langchain/short-term-memory and https://docs.langchain.com/oss/python/langchain/long-term-memory
  memory = ConversationBufferMemory(memory_key="chat_history")
/usr/local/lib/python3.13/dist-packages/pydantic/main.py:263: LangChainDeprecationWarning: The class `InMemoryChatMessageHistory` was deprecated in LangChain 1.6.4 and will be removed in 2.0.0 See the short-term memory documentation for recommended alternatives: https://docs.langchain.com/oss/python/langchain/short-term-memory
  validated_self = self.__pydantic_validator__.validate_python(data, self_instance=self)


In [25]:
# 간단한 질문을 하여 대화 기록을 만듭니다.
llm_chain.invoke({"input_prompt": "Hi! My name is Maarten. What is 1 + 1?"})

{'input_prompt': 'Hi! My name is Maarten. What is 1 + 1?',
 'chat_history': '',
 'text': ' Hi Maarten! 1 + 1 equals 2.'}

In [26]:
# LLM이 이름을 기억할까요?
llm_chain.invoke({"input_prompt": "What is my name?"})

{'input_prompt': 'What is my name?',
 'chat_history': 'Human: Hi! My name is Maarten. What is 1 + 1?\nAI:  Hi Maarten! 1 + 1 equals 2.',
 'text': ' Your name is the AI.\n\nHowever, if you are referring to the human in this conversation, my previous response indicated that their name is Maarten.'}

### 윈도 대화 버퍼

In [27]:
from langchain_classic.memory import ConversationBufferWindowMemory

# 메모리에 마지막 두 개의 대화만 유지합니다.
memory = ConversationBufferWindowMemory(k=2, memory_key="chat_history")

# LLM, 프롬프트, 메모리를 연결합니다.
llm_chain = LLMChain(
    prompt=prompt,
    llm=llm,
    memory=memory
)

/tmp/ipykernel_2277/40775140.py:4: LangChainDeprecationWarning: The class `ConversationBufferWindowMemory` was deprecated in LangChain 0.3.1 and will be removed in 2.0.0. Use `langchain.agents.create_agent` instead. For agents that need to remember prior interactions, use `create_agent` with checkpointing or the `Store` API. See https://docs.langchain.com/oss/python/langchain/short-term-memory and https://docs.langchain.com/oss/python/langchain/long-term-memory
  memory = ConversationBufferWindowMemory(k=2, memory_key="chat_history")


In [28]:
# 두 개의 질문을 던져 메모리에 대화 기록을 저장합니다.
llm_chain.invoke({"input_prompt":"Hi! My name is Maarten and I am 33 years old. What is 1 + 1?"})
llm_chain.invoke({"input_prompt":"What is 3 + 3?"})

{'input_prompt': 'What is 3 + 3?',
 'chat_history': "Human: Hi! My name is Maarten and I am 33 years old. What is 1 + 1?\nAI:  Hello Maarten! 1 + 1 equals 2. I hope you're having a great day at the age of 33!",
 'text': ' Hello Maarten! 3 + 3 equals 6. Have a wonderful day at the age of 33!'}

In [29]:
# 이름을 기억하는고 있는지 확인합니다.
llm_chain.invoke({"input_prompt":"What is my name?"})

{'input_prompt': 'What is my name?',
 'chat_history': "Human: Hi! My name is Maarten and I am 33 years old. What is 1 + 1?\nAI:  Hello Maarten! 1 + 1 equals 2. I hope you're having a great day at the age of 33!\nHuman: What is 3 + 3?\nAI:  Hello Maarten! 3 + 3 equals 6. Have a wonderful day at the age of 33!",
 'text': ' Your name is Maarten.'}

In [30]:
# 나이를 기억하는고 있는지 확인합니다.
llm_chain.invoke({"input_prompt":"What is my age?"})

{'input_prompt': 'What is my age?',
 'chat_history': 'Human: What is 3 + 3?\nAI:  Hello Maarten! 3 + 3 equals 6. Have a wonderful day at the age of 33!\nHuman: What is my name?\nAI:  Your name is Maarten.',
 'text': " As an AI, I don't have access to personal data like your age unless it has been shared in the context of our conversation. Therefore, I can't provide your actual age without that information. If you are referring to a specific date or scenario within this interaction where your age was mentioned hypothetically, please clarify so I can assist accordingly. Otherwise, for privacy reasons, I cannot access such personal data."}

### 대화 요약

In [31]:
# 요약 프롬프트 템플릿을 만듭니다.
summary_prompt_template = """<|user|>Summarize the conversations and update with the new lines.

Current summary:
{summary}

new lines of conversation:
{new_lines}

New summary:<|end|>
<|assistant|>"""
summary_prompt = PromptTemplate(
    input_variables=["new_lines", "summary"],
    template=summary_prompt_template
)

In [32]:
from langchain_classic.memory import ConversationSummaryMemory

# 사용할 메모리를 정의합니다.
memory = ConversationSummaryMemory(
    llm=llm,
    memory_key="chat_history",
    prompt=summary_prompt
)

# LLM, 프롬프트, 메모리를 연결합니다.
llm_chain = LLMChain(
    prompt=prompt,
    llm=llm,
    memory=memory
)

/tmp/ipykernel_2277/385226916.py:4: LangChainDeprecationWarning: The class `ConversationSummaryMemory` was deprecated in LangChain 0.3.1 and will be removed in 2.0.0. Use `langchain.agents.create_agent` instead. For agents that need to remember prior interactions, use `create_agent` with checkpointing or the `Store` API. See https://docs.langchain.com/oss/python/langchain/short-term-memory and https://docs.langchain.com/oss/python/langchain/long-term-memory
  memory = ConversationSummaryMemory(


In [33]:
# 이름에 대해 질문하는 대화를 생성합니다.
llm_chain.invoke({"input_prompt": "Hi! My name is Maarten. What is 1 + 1?"})
llm_chain.invoke({"input_prompt": "What is my name?"})

{'input_prompt': 'What is my name?',
 'chat_history': ' New summary: Maarten, a user, asked the AI to solve the basic arithmetic addition problem 1 + 1. The AI confirmed that their sum equals 2 and provided an explanation of combining two units together in this instance.',
 'text': ' The name mentioned in the current summary is Maarten. However, based on your question "What is my name?", it\'s not clear whose name you are referring to from this context. As an AI, I don\'t have personal information unless previously provided during our interaction. Could you please provide more details or clarify your query?'}

In [34]:
# 지금까지 내용이 요약되어 있는지 확인합니다.
llm_chain.invoke({"input_prompt": "What was the first question I asked?"})

{'input_prompt': 'What was the first question I asked?',
 'chat_history': " Maarten inquired about the sum of 1 + 1, and the AI confirmed it equals 2 by explaining that combining two units results in this value. Upon receiving a query regarding their name, the AI clarified its limitation as an AI without personal information but mentioned Maarten's name from previous interactions for context understanding.",
 'text': ' The first question you asked was, "What is the sum of 1 + 1?"'}

In [35]:
# 지금까지 요약 내용을 확인합니다.
memory.load_memory_variables({})

{'chat_history': ' Maarten initially asked for the sum of 1 + 1, which the AI confirmed as equal to 2 by explaining that combining two units results in this value. Later, when inquiring about their name, the AI clarified its limitation as an AI without personal information but mentioned Maarten\'s name from previous interactions for context understanding. Upon being asked what was the first question they asked, the human learned it was "What is the sum of 1 + 1?"'}

# 에이전트

In [36]:
import os
from langchain_openai import ChatOpenAI

# 랭체인으로 오픈AI의 LLM을 로드합니다.
os.environ["OPENAI_API_KEY"] = "MY_KEY"
openai_llm = ChatOpenAI(model_name="gpt-3.5-turbo", temperature=0)

In [37]:
# ReAct 템플릿을 만듭니다.
react_template = """Answer the following questions as best you can. You have access to the following tools:

{tools}

Use the following format:

Question: the input question you must answer
Thought: you should always think about what to do
Action: the action to take, should be one of [{tool_names}]
Action Input: the input to the action
Observation: the result of the action
... (this Thought/Action/Action Input/Observation can repeat N times)
Thought: I now know the final answer
Final Answer: the final answer to the original input question

Begin!

Question: {input}
Thought:{agent_scratchpad}"""

prompt = PromptTemplate(
    template=react_template,
    input_variables=["tools", "tool_names", "input", "agent_scratchpad"]
)

In [38]:
from langchain_core.tools import Tool
from langchain_community.agent_toolkits.load_tools import load_tools
from langchain_classic.tools import DuckDuckGoSearchResults

# 에이전트에 전달할 도구를 만듭니다.
search = DuckDuckGoSearchResults()
search_tool = Tool(
    name="duckduck",
    description="A web search engine. Use this to as a search engine for general queries.",
    func=search.run,
)

# 도구를 준비합니다.
tools = load_tools(["llm-math"], llm=openai_llm)

# 기존 'Calculator' 도구의 설명을 더 구체적으로 덮어씁니다.
for tool in tools:
    if tool.name == "Calculator":
        tool.description = (
            "Useful for when you need to answer questions about math. "
            "Input should be a strictly numerical mathematical expression. "
            "Do NOT include any words, variable names, or currency symbols like 'price' or '$'."
        )

tools.append(search_tool)

ModuleNotFoundError: No module named 'mypy_extensions'

In [ ]:
from langchain_classic.agents import AgentExecutor, create_react_agent

# ReAct 에이전트를 만듭니다.
agent = create_react_agent(openai_llm, tools, prompt)
agent_executor = AgentExecutor(
    agent=agent, tools=tools, verbose=True, handle_parsing_errors=True
)

In [ ]:
# 맥북 프로의 가격은 얼마인가요?
agent_executor.invoke(
    {
        "input": "What is the current price of a MacBook Pro in USD? How much would it cost in EUR if the exchange rate is 0.85 EUR for 1 USD?"
    }
)